# Filter & Reduce Dataset — RAG Soạn thảo Văn bản Hành chính

Notebook này thêm **bước 1.5** vào pipeline, chạy **sau** `preprocessData.ipynb` và **trước** `chunkData.ipynb`.

## Mục tiêu
Giảm `legal_dataset_processed.parquet` từ ~392k hàng xuống ~110k hàng bằng cách chỉ giữ lại các loại văn bản và văn bản cốt lõi thực sự cần thiết cho RAG soạn thảo hành chính. Kết quả sau khi re-embed:

| Thành phần | Trước | Sau |
|---|---|---|
| legal chunks | 593,430 | ~165,000 |
| ChromaDB | ~9.4 GB | ~2.6 GB |
| BM25 pkl | ~400 MB | ~110 MB |
| **Tổng** | **~11.5 GB** | **~3 GB** |

## Vị trí trong pipeline
```
preprocessData.ipynb   → legal_dataset_processed.parquet (392k hàng)
       ↓
filterData.ipynb       → legal_dataset_filtered.parquet  (~110k hàng)  ← notebook này
       ↓
chunkData.ipynb        → legal_chunks.parquet (~165k chunks)
       ↓
embedAndIndex.ipynb    → ChromaDB (~2.6 GB)
```

> `forms_dataset_processed.parquet` và `forms_examples_dataset_processed.parquet` **giữ nguyên** — đã đủ nhỏ.


## 0. Setup

In [2]:
import pandas as pd
import re
from pathlib import Path

DATASET_DIR = Path("dataset/processed")

INPUT_PATH  = DATASET_DIR / "legal_dataset_processed.parquet"
OUTPUT_PATH = DATASET_DIR / "legal_dataset_filtered.parquet"

df = pd.read_parquet(INPUT_PATH, engine="pyarrow")
print(f"Input: {len(df):,} hàng | cột: {list(df.columns)}")
print(f"\nPhân bố type_normalized:")
print(df["type_normalized"].value_counts().to_string())


Input: 392,079 hàng | cột: ['doc_id', 'id', 'ministry', 'type_normalized', 'name', 'chapter_id', 'chapter_name', 'article', 'content']

Phân bố type_normalized:
type_normalized
THÔNG TƯ        132431
QUYẾT ĐỊNH       85733
NGHỊ ĐỊNH        71176
NGHỊ QUYẾT       59827
LUẬT             31145
PHÁP LỆNH         6798
KHÁC              4859
VĂN BẢN KHÁC        83
KHÔNG RÕ            27


## 1. Định nghĩa Filter

### Chiến lược giữ lại
1. **Theo loại văn bản:** `LUẬT`, `NGHỊ ĐỊNH`, `PHÁP LỆNH` — đây là 3 loại quy phạm pháp luật có giá trị pháp lý cao nhất, đủ để cover mọi nhu cầu soạn thảo hành chính cơ bản
2. **Văn bản cốt lõi bổ sung:** Một số `THÔNG TƯ` và `QUYẾT ĐỊNH` quan trọng trực tiếp liên quan đến soạn thảo văn bản hành chính (chủ yếu là NĐ 30/2020 và các hướng dẫn thi hành)

### Lý do không giữ toàn bộ THÔNG TƯ / QUYẾT ĐỊNH
- `THÔNG TƯ` chiếm 33.8% (~200k điều) — phần lớn là hướng dẫn kỹ thuật chuyên ngành không liên quan đến soạn thảo hành chính chung
- `QUYẾT ĐỊNH` chiếm 21.9% (~130k điều) — phần lớn là quyết định hành chính cụ thể từng vụ việc, không phải quy phạm


In [3]:
# ── Loại văn bản giữ lại ────────────────────────────────────────────────────
KEEP_TYPES = {
    "LUẬT",        # 7.9% — quy phạm cao nhất
    "NGHỊ ĐỊNH",   # 18.2% — hướng dẫn thi hành luật
    "PHÁP LỆNH",   # 1.7%  — có giá trị như luật
    "NGHỊ QUYẾT",  # 15.3% — chính sách cấp cao
}

# ── Số hiệu văn bản cốt lõi bổ sung (THÔNG TƯ/QĐ quan trọng) ───────────────
# Chỉ giữ các văn bản TRỰC TIẾP liên quan đến công tác văn thư, soạn thảo
CORE_DOC_NOS = [
    "30/2020/NĐ-CP",      # Công tác văn thư — quan trọng nhất
    "01/2011/QH13",        # Luật Lưu trữ
    "78/2019/TT-BTC",     # Hướng dẫn chế độ công văn (nếu có)
]

# Pattern match linh hoạt cho source_doc_no
CORE_PATTERN = "|".join(re.escape(d) for d in CORE_DOC_NOS)

print("Giữ lại:")
print(f"  Types: {KEEP_TYPES}")
print(f"  Core docs: {CORE_DOC_NOS}")


Giữ lại:
  Types: {'PHÁP LỆNH', 'NGHỊ ĐỊNH', 'LUẬT', 'NGHỊ QUYẾT'}
  Core docs: ['30/2020/NĐ-CP', '01/2011/QH13', '78/2019/TT-BTC']


## 2. Áp dụng Filter

In [4]:
n_before = len(df)

# Mask 1: theo loại văn bản
mask_type = df["type_normalized"].isin(KEEP_TYPES)

# Mask 2: văn bản cốt lõi bổ sung (giữ dù thuộc type nào)
# Dùng cột "id" (số hiệu văn bản gốc) để match
if "id" in df.columns:
    mask_core = df["id"].str.contains(CORE_PATTERN, na=False, regex=True)
else:
    mask_core = pd.Series(False, index=df.index)

df_filtered = df[mask_type | mask_core].reset_index(drop=True)
n_after = len(df_filtered)

print(f"Trước filter : {n_before:,} hàng")
print(f"Sau filter   : {n_after:,} hàng")
print(f"Đã loại bỏ  : {n_before - n_after:,} hàng ({(n_before - n_after)/n_before*100:.1f}%)")
print()
print("Phân bố type_normalized sau filter:")
print(df_filtered["type_normalized"].value_counts().to_string())


Trước filter : 392,079 hàng
Sau filter   : 168,946 hàng
Đã loại bỏ  : 223,133 hàng (56.9%)

Phân bố type_normalized sau filter:
type_normalized
NGHỊ ĐỊNH     71176
NGHỊ QUYẾT    59827
LUẬT          31145
PHÁP LỆNH      6798


## 3. Kiểm tra Coverage — Văn bản cốt lõi

In [5]:
# Đảm bảo 8 văn bản quan trọng nhất vẫn còn đầy đủ
IMPORTANT_LAWS = {
    "Nghị định 30/2020/NĐ-CP (Công tác văn thư)" : r"30/2020/NĐ-CP",
    "Bộ luật Lao động 2019"                        : r"45/2019/QH14",
    "Luật Cán bộ, Công chức"                       : r"Luật Cán bộ, công chức",
    "Luật Viên chức"                               : r"Luật Viên chức",
    "Luật Ban hành văn bản QPPL"                   : r"80/2015/QH13|15/2020/QH14",
    "Luật Lưu trữ"                                 : r"01/2011/QH13",
    "Luật Tổ chức Chính phủ"                       : r"Luật Tổ chức Chính phủ",
    "Luật Tổ chức Chính quyền địa phương"          : r"Luật Tổ chức chính quyền địa phương",
}

print(f"{'Văn bản':<50} {'Điều khoản':>12} {'Trạng thái'}")
print("-" * 75)

id_col = "id" if "id" in df_filtered.columns else "source_doc_no"
all_ok = True
for label, pattern in IMPORTANT_LAWS.items():
    hits = df_filtered[id_col].str.contains(pattern, case=False, na=False).sum()
    # Fallback: tìm trong cột name nếu không có trong id
    if hits == 0 and "name" in df_filtered.columns:
        hits = df_filtered["name"].str.contains(pattern, case=False, na=False).sum()
    status = "✅" if hits > 0 else "❌ MISSING"
    if hits == 0:
        all_ok = False
    print(f"  {label:<48} {hits:>12,}   {status}")

print()
if all_ok:
    print("✅ Tất cả văn bản cốt lõi đều có mặt sau filter.")
else:
    print("⚠️  Một số văn bản cốt lõi bị thiếu — kiểm tra lại KEEP_TYPES hoặc CORE_DOC_NOS.")


Văn bản                                              Điều khoản Trạng thái
---------------------------------------------------------------------------
  Nghị định 30/2020/NĐ-CP (Công tác văn thư)                 63   ✅
  Bộ luật Lao động 2019                                     220   ✅
  Luật Cán bộ, Công chức                                    109   ✅
  Luật Viên chức                                             43   ✅
  Luật Ban hành văn bản QPPL                                174   ✅
  Luật Lưu trữ                                               42   ✅
  Luật Tổ chức Chính phủ                                      3   ✅
  Luật Tổ chức Chính quyền địa phương                         3   ✅

✅ Tất cả văn bản cốt lõi đều có mặt sau filter.


## 4. Kiểm tra nhanh chất lượng

In [6]:
# Missing values
print("Missing values sau filter:")
miss = df_filtered.isnull().sum()
print(miss[miss > 0].to_string() if miss.sum() > 0 else "  ✅ Không có missing values")
print()

# Duplicate doc_id
dup = df_filtered["doc_id"].duplicated().sum()
print(f"Duplicate doc_id: {dup}")
if dup == 0:
    print("  ✅ Không có duplicate doc_id")
print()

# Word count distribution (ước tính)
df_filtered["_wc"] = df_filtered["content"].fillna("").apply(lambda x: len(x.split()))
print("Phân bố độ dài content (số từ):")
print(df_filtered["_wc"].describe().round(1).to_string())
df_filtered = df_filtered.drop(columns=["_wc"])


Missing values sau filter:
id            158
chapter_id      3

Duplicate doc_id: 0
  ✅ Không có duplicate doc_id

Phân bố độ dài content (số từ):
count    168946.0
mean        302.8
std         568.2
min           1.0
25%          85.0
50%         165.0
75%         320.0
max       55368.0


## 5. Lưu Output

In [7]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_filtered.to_parquet(OUTPUT_PATH, engine="pyarrow", index=False)

size_mb = OUTPUT_PATH.stat().st_size / 1024 / 1024
print(f"✅ Đã lưu: {OUTPUT_PATH}")
print(f"   Số hàng    : {len(df_filtered):,}")
print(f"   Số cột     : {len(df_filtered.columns)}")
print(f"   Dung lượng : {size_mb:.1f} MB")
print()
print("Bước tiếp theo: Chạy chunkData.ipynb với INPUT_PATH trỏ vào file này.")
print(f"  INPUT_PATH = Path('dataset/processed/legal_dataset_filtered.parquet')")


✅ Đã lưu: dataset\processed\legal_dataset_filtered.parquet
   Số hàng    : 168,946
   Số cột     : 9
   Dung lượng : 109.8 MB

Bước tiếp theo: Chạy chunkData.ipynb với INPUT_PATH trỏ vào file này.
  INPUT_PATH = Path('dataset/processed/legal_dataset_filtered.parquet')


## 6. Tóm tắt thay đổi cần làm ở các notebook tiếp theo

Sau khi chạy notebook này, cần sửa **1 dòng** trong `chunkData.ipynb`:

```python
# TRƯỚC (Cell 2 — Input paths)
"legal" : DATASET_DIR / "legal_dataset_processed.parquet",

# SAU
"legal" : DATASET_DIR / "legal_dataset_filtered.parquet",
```

Các notebook khác (`embedAndIndex.ipynb`, `hybridRetrieval.ipynb`) **không cần sửa** vì chúng đọc từ `chunks/` chứ không phải từ `processed/`.
